# 05 — Validation complémentaire par mutations

Validation contrôlée par **injection de fautes** sur 4 modules (account, stock, sale, purchase). Pour chaque mutation, on vérifie si le modèle remonte les tests réellement cassés en tête (**APFD**). Les échecs d'environnement pré-existants sont **filtrés**.

**Entrée** : `artifacts/dataset_features.pkl`, `artifacts/embeddings_sbert.pkl`, `models/ranker_full.pkl`, `data/mutations/*.log`.

In [1]:
# ── Chemins relatifs au package (portable) ──
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == 'scripts': ROOT = ROOT.parent
DATA = ROOT / 'data'
ARTIFACTS = ROOT / 'artifacts'
MODELS = ROOT / 'models'
print('Racine du package :', ROOT)

Racine du package : c:\Users\LEGION\OneDrive\Bureau\package_evaluation


In [2]:
import pandas as pd, numpy as np, re, pickle, joblib, torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics.pairwise import cosine_similarity
# Désactive les avertissements non critiques et fixe la graine (reproductibilité)
import warnings; warnings.filterwarnings('ignore'); np.random.seed(42)

# Chargement des artefacts : dataset enrichi en features, embeddings SBERT,
# graphe de dépendances inverses, mots génériques et liste des features.
df = pd.read_pickle(ARTIFACTS/'dataset_features.pkl')
emb = pickle.load(open(ARTIFACTS/'embeddings_sbert.pkl','rb'))
test_emb_sbert = emb['test_emb_sbert']
reverse_deps = pickle.load(open(ARTIFACTS/'reverse_deps.pkl','rb'))
GENERIC_WORDS = pickle.load(open(ARTIFACTS/'generic.pkl','rb'))
FEATURES = pickle.load(open(MODELS/'feature_list.pkl','rb'))

# ── Chargement des 3 modèles finaux entraînés (notebook 03) ──
# XGBoost et LightGBM se chargent directement via joblib.
xgb_model  = joblib.load(MODELS/'xgboost_ranker.pkl')
lgbm_model = joblib.load(MODELS/'lightgbm_ranker.pkl')

# RankNet : un modèle PyTorch nécessite de redéfinir d'abord l'architecture
# (la classe), puis de charger les poids sauvegardés et le scaler associé.
class RankNetBase(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

ranknet_model = RankNetBase(len(FEATURES))
ranknet_model.load_state_dict(torch.load(MODELS/'ranknet_model.pt'))
ranknet_model.eval()  # mode évaluation (désactive le Dropout)
ranknet_scaler = joblib.load(MODELS/'ranknet_scaler.pkl')

# Fonctions de prédiction uniformes pour les 3 modèles. RankNet nécessite
# en plus la normalisation des features (via son scaler) et le mode sans gradient.
def predict_xgb(X):  return xgb_model.predict(X)
def predict_lgbm(X): return lgbm_model.predict(X)
def predict_ranknet(X):
    with torch.no_grad():
        return ranknet_model(torch.FloatTensor(ranknet_scaler.transform(X))).numpy()

# Dictionnaire {nom: fonction de prédiction} parcouru lors de la comparaison
MODELS_TO_TEST = {
    'XGBoost':  predict_xgb,
    'LightGBM': predict_lgbm,
    'RankNet':  predict_ranknet,
}

from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print(' Setup OK :', df.shape, '| 3 modèles chargés')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Setup OK : (263649, 31) | 3 modèles chargés


In [3]:
# ── Lecture des logs + extraction des tests cassés ──
def read_log(path):
    """Lit un log de mutation en gérant son encodage (UTF-16/UTF-8/Latin-1)."""
    with open(path,'rb') as f: raw=f.read()
    if raw[:2]==b'\xff\xfe': return raw.decode('utf-16-le',errors='replace')
    try: return raw.decode('utf-8',errors='replace')
    except: return raw.decode('latin-1',errors='replace')


def extract_broken_tests(log_path, env_error_classes=None):
    """Extrait les tests cassés d'un log de mutation, hors bruit d'environnement.

    Recherche dans le log les tests en échec (FAIL/ERROR/FAILED) et retourne
    leur identifiant. Les classes de test connues pour échouer indépendamment
    de la mutation (bruit d'environnement) sont exclues afin de ne conserver
    que les échecs réellement imputables à la faute injectée.

    Args:
        log_path: Chemin du log d'exécution après mutation.
        env_error_classes: Ensemble de classes de test à ignorer (bruit).

    Returns:
        Liste triée des identifiants de tests cassés (classe.méthode).
    """
    text = read_log(log_path)
    env_classes = env_error_classes or set()
    broken = set()
    for pattern in [r'(?:FAIL|ERROR):\s+(\w+\.\w+)', r'FAILED\s+(\w+\.\w+)']:
        for m in re.finditer(pattern, text):
            tn=m.group(1); cn=tn.split('.')[0]
            if cn not in env_classes: broken.add(tn)
    return sorted(broken)


# Classes de test à exclure : elles échouent pour des raisons liées à
# l'environnement (configuration, accès fichiers, PDF, profiling, etc.) et non
# à la mutation. Les filtrer évite de compter de faux tests cassés.
ENV_NOISE = {
    'TestConfigManager','TestAddonsFileAccess','TestCommand','TestClocFields',
    'TestProfiling','TestSelector','TestSelectorSelection','TestAuditTrail',
    'TestAuditTrailAttachment','TestReports','TestSpreadsheet','TestSpreadsheetUtils',
    'TestUrl','TestUrlJoin','TestPdf','TestPhonenumbers','TestQWebBasic',
    'TestImportModule','TestImportFiles','TestBaseDocumentLayout',
    'TestAggregatePdfReports','TestIrActionsReport','TestPackagePropagation',
    'test_guess_mimetype','TestUblImportBis3InvoiceBEAutoGeneratePDF',
}
print('Helpers + filtre de bruit définis')

Helpers + filtre de bruit définis


In [5]:
# ── Configuration des 4 mutations ──
MUT_DIR = DATA / 'mutations'

MUTATIONS = {
    'mutation1_account': {
        'description': "amount_untaxed *= 1.1 dans account_move.py", 'module': 'account',
        'log': str(MUT_DIR / 'MUTATION1.log'),
        'fake_diff': """diff --git a/addons/account/models/account_move.py b/addons/account/models/account_move.py
@@ -1215,7 +1215,7 @@
-            move.amount_untaxed = sign * total_untaxed_currency
+            move.amount_untaxed = sign * total_untaxed_currency * 1.1""",
    },
    'mutation2_stock': {
        'description': "product_qty *= 2 dans stock_move.py", 'module': 'stock',
        'log': str(MUT_DIR / 'MUTATION2_stock.log'),
        'fake_diff': """diff --git a/addons/stock/models/stock_move.py b/addons/stock/models/stock_move.py
@@ -379,7 +379,7 @@
-            move.product_qty = move.product_uom._compute_quantity(move.product_uom_qty, move.product_id.uom_id, rounding_method='HALF-UP')
+            move.product_qty = move.product_uom._compute_quantity(move.product_uom_qty, move.product_id.uom_id, rounding_method='HALF-UP') * 2""",
    },
    'mutation3_sale': {
        'description': "price_subtotal *= 0.8 dans sale_order_line.py", 'module': 'sale',
        'log': str(MUT_DIR / 'MUTATION3_sale.log'),
        'fake_diff': """diff --git a/addons/sale/models/sale_order_line.py b/addons/sale/models/sale_order_line.py
@@ -851,7 +851,7 @@
-            line.price_subtotal = base_line['tax_details']['total_excluded_currency']
+            line.price_subtotal = base_line['tax_details']['total_excluded_currency'] * 0.8""",
    },
    'mutation4_purchase': {
        'description': "price_subtotal *= 0.9 dans purchase_order_line.py", 'module': 'purchase',
        'log': str(MUT_DIR / 'MUTATION4_purchase.log'),
        'fake_diff': """diff --git a/addons/purchase/models/purchase_order_line.py b/addons/purchase/models/purchase_order_line.py
@@ -159,7 +159,7 @@
-            line.price_subtotal = base_line['tax_details']['total_excluded_currency']
+            line.price_subtotal = base_line['tax_details']['total_excluded_currency'] * 0.9""",
    },
}
print(f" {len(MUTATIONS)} mutations configurées")

 4 mutations configurées


## Évaluation des mutations + APFD

In [6]:
# ── Référence : commit avec ~8700 tests (terrain de jeu à classer) ──
# On part d'un commit dense (commit10) dont on connaît tous les tests, puis on
# y simule chaque mutation : le modèle doit replacer en tête les tests cassés.
ref_commit = 'commit10'
df_ref_base = df[df.commit_id == ref_commit].copy()

all_model_results = {}

# Boucle externe : chaque modèle de ranking est évalué sur toutes les mutations
for model_name, predict_fn in MODELS_TO_TEST.items():
    print(f"\n{'='*70}\nMODÈLE : {model_name}\n{'='*70}")
    summary_results = []

    # Boucle interne : chaque mutation injectée dans un module donné
    for mut_name, mut_info in MUTATIONS.items():
        if not os.path.exists(mut_info['log']):
            continue
        # Tests réellement cassés par la mutation (hors bruit d'environnement)
        broken_tests = extract_broken_tests(mut_info['log'], env_error_classes=ENV_NOISE)
        if len(broken_tests) == 0:
            continue

        # Recalcule les features liées à la mutation comme s'il s'agissait d'un
        # vrai commit : similarité SBERT au faux diff, module touché, dépendances.
        fake_diff_emb = sbert.encode([mut_info['fake_diff']], convert_to_numpy=True)[0]
        df_ref = df_ref_base.copy()
        df_ref['similarity_score'] = [
            float(cosine_similarity([fake_diff_emb],
                [test_emb_sbert.get(t, np.zeros(384))])[0][0])
            for t in df_ref['test_name']]

        mut_module = mut_info['module']
        df_ref['module_touched'] = df_ref['module'].apply(lambda m: int(m == mut_module))
        df_ref['module_dependency_touched'] = df_ref['module'].apply(
            lambda m: int(m in reverse_deps.get(mut_module, set())))
        # Métadonnées de diff fixées arbitrairement (la mutation est un petit changement)
        df_ref['commit_nb_files'] = 1
        df_ref['commit_lines_added'] = 5
        df_ref['commit_lines_deleted'] = 5
        df_ref['commit_lines_total'] = 10
        df_ref['test_file_touched'] = 0
        df_ref['filename_kw_count'] = 0
        df_ref['has_filename_kw'] = 0

        # Prédiction et tri des tests selon le modèle courant
        X_fake = df_ref[FEATURES].values
        df_ref['score_mut'] = predict_fn(X_fake)

        df_sorted = df_ref.sort_values('score_mut', ascending=False).reset_index(drop=True)
        total = len(df_sorted)
        # Marque les tests effectivement cassés dans le classement
        df_sorted['is_broken'] = df_sorted['test_name'].apply(lambda t: int(t in broken_tests))
        found = [t for t in broken_tests if t in df_ref['test_name'].values]

        # Positions (en %) des tests cassés dans le classement produit
        positions = [(i+1)/total*100 for i, r in df_sorted.iterrows() if r['is_broken'] == 1]
        if not positions:
            continue

        # Indicateurs : combien de tests cassés dans le top 5 % / 10 %, position
        # moyenne, et APFD (capacité à remonter tôt les tests cassés).
        top5  = sum(1 for p in positions if p <= 5)
        top10 = sum(1 for p in positions if p <= 10)
        avg   = np.mean(positions)
        ordered = df_sorted['is_broken'].tolist()
        n, m = len(ordered), sum(ordered)
        pos_list = [i+1 for i, l in enumerate(ordered) if l == 1]
        apfd = 1 - sum(pos_list)/(n*m) + 1/(2*n) if m > 0 else 0.5

        summary_results.append({'mutation': mut_name, 'module': mut_module,
            'broken': len(broken_tests), 'found': len(found),
            'apfd': apfd, 'top5': top5, 'top10': top10, 'avg_pct': avg})
        print(f"  {mut_name:<22} {mut_module:<10} cassés={len(broken_tests):>4}  "
            f"trouvés={len(found):>4}  APFD={apfd:.4f}  top10%={top10}/{len(positions)}")

    # APFD moyen du modèle sur l'ensemble des mutations
    mean_apfd = np.mean([r['apfd'] for r in summary_results]) if summary_results else 0.0
    all_model_results[model_name] = {'details': summary_results, 'mean_apfd': mean_apfd}
    print(f"  {'→ APFD MOYEN':<33} {mean_apfd:.4f}")

# ── Tableau comparatif final des 3 modèles (APFD par mutation) ──
print('COMPARAISON DES 3 MODÈLES SUR LES MUTATIONS (APFD)')
print(f"\n  {'Mutation':<22} {'XGBoost':>9} {'LightGBM':>9} {'RankNet':>9}")

muts = [r['mutation'] for r in all_model_results['XGBoost']['details']]
for mut in muts:
    row = f"  {mut:<22}"
    for mdl in ['XGBoost', 'LightGBM', 'RankNet']:
        apfd = next((r['apfd'] for r in all_model_results[mdl]['details']
                    if r['mutation'] == mut), 0.0)
        row += f"{apfd:>9.4f}"
    print(row)

row = f"  {'APFD MOYEN':<22}"
for mdl in ['XGBoost', 'LightGBM', 'RankNet']:
    row += f"{all_model_results[mdl]['mean_apfd']:>9.4f}"
print(row)


MODÈLE : XGBoost
  mutation1_account      account    cassés= 290  trouvés= 290  APFD=0.8785  top10%=185/290
  mutation2_stock        stock      cassés= 178  trouvés= 176  APFD=0.7849  top10%=64/176
  mutation3_sale         sale       cassés=  59  trouvés=  54  APFD=0.8143  top10%=17/54
  mutation4_purchase     purchase   cassés=  48  trouvés=  47  APFD=0.7557  top10%=8/47
  → APFD MOYEN                      0.8083

MODÈLE : LightGBM
  mutation1_account      account    cassés= 290  trouvés= 290  APFD=0.8950  top10%=152/290
  mutation2_stock        stock      cassés= 178  trouvés= 176  APFD=0.8070  top10%=62/176
  mutation3_sale         sale       cassés=  59  trouvés=  54  APFD=0.7730  top10%=20/54
  mutation4_purchase     purchase   cassés=  48  trouvés=  47  APFD=0.7368  top10%=7/47
  → APFD MOYEN                      0.8029

MODÈLE : RankNet
  mutation1_account      account    cassés= 290  trouvés= 290  APFD=0.8932  top10%=160/290
  mutation2_stock        stock      cassés= 178  tro